# Dodgers 2027 model and predictions

This notebook reviews the production forecast generated for the Dodgers 2027 roster-planning application. The selected models were refit using data through 2026 after the 2026 holdout evaluation. Because 2027 outcomes are not yet available, these are forecasts rather than validated results.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
forecast_path = ROOT / 'data' / 'forecasts' / 'player_forecasts_2027.csv'
dodgers_path = ROOT / 'data' / 'forecasts' / 'dodgers_forecast_shortlist_2027.csv'
forecast = pd.read_csv(forecast_path)
dodgers = pd.read_csv(dodgers_path)
print(f'All forecast rows: {len(forecast):,}')
print(f'Dodgers shortlist rows: {len(dodgers):,}')

All forecast rows: 407
Dodgers shortlist rows: 13


## Forecast audit

The production output should contain one row per player, no 2027 outcomes, and complete Statcast flags for the scoring population.

In [2]:
required = ['Name', 'season', 'forecast_season', 'pa_fg', 'predicted_eligibility_probability', 'predicted_wrc_plus', 'statcast_missing']
missing_columns = sorted(set(required) - set(forecast.columns))
assert not missing_columns, f'Missing columns: {missing_columns}'
assert forecast['Name'].notna().all()
assert forecast['mlbam_id'].notna().all()
assert not forecast['mlbam_id'].duplicated().any(), 'Forecast rows must have unique MLBAM IDs'
assert (forecast['forecast_season'] == 2027).all()
print(f'Statcast-missing rows: {forecast.statcast_missing.sum()} / {len(forecast)}')
forecast[required].head()

AssertionError: 

## Dodgers ranked shortlist

The table is ordered by the production shortlist ranking. Eligibility is the modeled probability of reaching at least 150 PA; predicted wRC+ is the conditional offensive forecast.

In [ ]:
display_columns = [
    'Name', 'pa_fg', 'wrc_plus', 'predicted_wrc_plus',
    'predicted_eligibility_probability', 'predicted_wrc_plus_lift_vs_2026'
]
ranked = dodgers[display_columns].copy()
ranked.columns = ['Player', '2026 PA', '2026 wRC+', 'Predicted 2027 wRC+', 'Eligibility probability', 'Lift vs 2026']
ranked.round(1)

In [ ]:
plot_df = ranked.sort_values('Predicted 2027 wRC+').tail(8)
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(plot_df['Player'], plot_df['Predicted 2027 wRC+'], color='#0b1f3a')
ax.set_xlabel('Predicted 2027 wRC+')
ax.set_title('Dodgers 2027 predicted wRC+ shortlist')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.show()

## Interpretation

Use the shortlist as a starting point for Dodgers baseball review. The model does not incorporate contracts, injuries, roster moves, defensive value, or expected lineup role, so those factors should be evaluated before any decision.